# Supervisor Node — Interactive Tests

The **supervisor** is a pure routing function — it reads the current `AgentState` and returns the name of the next node LangGraph should run.

It never modifies state. It is used as a **conditional edge** in the graph:

```
START → supervisor → planner
                   → executor
                   → END
```

### Routing rules
| Condition | Next node |
|---|---|
| `status` is `"done"` or `"error"` | `END` |
| `plan` list is empty | `"planner"` |
| `status` is `"executing"` | `"executor"` |
| anything else | `"planner"` (default) |

In [1]:
import sys
sys.path.insert(0, '../app/backend')  # so we can import from project root

from langgraph.graph import END
from nodes.supervisor import supervisor

## Helper
Builds a minimal valid `AgentState` so we only need to specify the fields we care about.

In [2]:
def make_state(status='planning', plan=None, task=''):
    """Return a minimal AgentState dict for testing."""
    return {
        'messages': [],
        'task': task,
        'plan': plan or [],
        'status': status,
        'results': [],
        'ui_events': [],
    }

## Case 1 — Task is done → `END`

In [3]:
state = make_state(status='done', plan=['step 1', 'step 2'])
result = supervisor(state)

print('Input status :', state['status'])
print('Input plan   :', state['plan'])
print('Routed to    :', result)
print('Is END?      :', result == END)

Input status : done
Input plan   : ['step 1', 'step 2']
Routed to    : __end__
Is END?      : True


## Case 2 — Error → `END`

In [4]:
state = make_state(status='error')
result = supervisor(state)

print('Input status :', state['status'])
print('Routed to    :', result)
print('Is END?      :', result == END)

Input status : error
Routed to    : __end__
Is END?      : True


## Case 3 — No plan yet → `"planner"`

In [5]:
state = make_state(status='planning', plan=[])
result = supervisor(state)

print('Input status :', state['status'])
print('Input plan   :', state['plan'])
print('Routed to    :', result)

Input status : planning
Input plan   : []
Routed to    : planner


## Case 4 — Has a plan and is executing → `"executor"`

In [6]:
state = make_state(status='executing', plan=['search the web', 'summarise results'])
result = supervisor(state)

print('Input status :', state['status'])
print('Input plan   :', state['plan'])
print('Routed to    :', result)

Input status : executing
Input plan   : ['search the web', 'summarise results']
Routed to    : executor


## Case 5 — Unknown status (fallback) → `"planner"`

In [7]:
state = make_state(status='something_unexpected', plan=['step 1'])
result = supervisor(state)

print('Input status :', state['status'])
print('Routed to    :', result)

Input status : something_unexpected
Routed to    : planner


## All cases at a glance

In [8]:
cases = [
    ('done + plan',         make_state(status='done',      plan=['s1'])),
    ('error',               make_state(status='error')),
    ('planning + no plan',  make_state(status='planning',  plan=[])),
    ('executing + plan',    make_state(status='executing', plan=['s1'])),
    ('unknown + plan',      make_state(status='???',       plan=['s1'])),
]

print(f"{'Case':<25} {'status':<12} {'plan?':<8} {'→'}")
print('-' * 55)
for label, s in cases:
    out = supervisor(s)
    print(f"{label:<25} {s['status']:<12} {bool(s['plan'])!s:<8} {out}")

Case                      status       plan?    →
-------------------------------------------------------
done + plan               done         True     __end__
error                     error        False    __end__
planning + no plan        planning     False    planner
executing + plan          executing    True     executor
unknown + plan            ???          True     planner
